# 🎵 Waveform Studio — Audio Visualizer Video Generator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mnchrmXD/waveform-studio/blob/main/waveform_studio.ipynb)

Generate studio-grade, audio-reactive visualizer videos directly in Google Colab using **Waveform Studio** (`mnchrmXD/waveform-studio`).

### Quick Flow:
1. **Launch Server**: Run Cell 1 to start the high-speed Node.js + FFmpeg headless rendering engine on `http://localhost:3000`.
2. **Upload Payload**: Run Cell 2 to upload your `payload.json` exported from Waveform Studio (or skip to use built-in defaults).
3. **Render Video**: Run Cell 3 to encode with hardware acceleration and stream the video.
4. **Preview & Download**: Run Cells 4 & 5 to preview in-notebook and download.

## 1. Setup Environment & Launch Server
Prepares dependencies and launches the Waveform Studio headless rendering engine in Colab in the background on `http://localhost:3000`.

In [ ]:
# Install Python utilities
!pip install -q requests tqdm

import os
import sys
import time
import json
import subprocess
import requests
from tqdm import tqdm
from IPython.display import HTML, Video, display

# 1. Ensure repository is available and active directory
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    if not os.path.exists('server.ts') and not os.path.exists('waveform-studio'):
        print("📥 Cloning Waveform Studio repository...")
        !git clone --depth 1 https://github.com/mnchrmXD/waveform-studio.git
        %cd waveform-studio
    elif os.path.exists('waveform-studio') and not os.path.exists('server.ts'):
        %cd waveform-studio

# 2. Fast install Node.js dependencies if not already present
if not os.path.exists('node_modules'):
    print("⚡ Installing Node.js & FFmpeg rendering dependencies...")
    !npm install --prefer-offline --no-audit --no-fund --loglevel=error

# 3. Check if server is already running on port 3000
API_URL = "http://localhost:3000"
server_ready = False

try:
    health_resp = requests.get(f"{API_URL}/api/health", timeout=2)
    if health_resp.status_code == 200:
        server_ready = True
        print("✅ Waveform Studio server is already running!")
except Exception:
    pass

# 4. Launch headless server in the background (HEADLESS_ONLY skips Vite frontend for instant start)
if not server_ready:
    print("🚀 Starting Waveform Studio headless rendering engine...")
    env = os.environ.copy()
    env["HEADLESS_ONLY"] = "true"
    env["NODE_ENV"] = "production"
    
    server_proc = subprocess.Popen(
        ["npx", "tsx", "server.ts"],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    
    # Poll until server is ready
    start_wait = time.time()
    for _ in range(30):
        try:
            health_resp = requests.get(f"{API_URL}/api/health", timeout=1)
            if health_resp.status_code == 200:
                server_ready = True
                print(f"✅ Waveform Studio server ready in {time.time() - start_wait:.1f}s on {API_URL}!")
                print(json.dumps(health_resp.json(), indent=2))
                break
        except Exception:
            time.sleep(0.5)

    if not server_ready:
        print("⚠️ Server initialization taking longer than expected. Check server output if needed.")

## 2. Upload Payload (JSON)
Upload your `payload.json` exported from Waveform Studio's **Payload Generator**.
*(If you skip this or don't upload a file, the renderer will automatically use default settings.)*

In [ ]:
payload = {}

try:
    from google.colab import files
    print("📤 Upload your payload.json (or cancel/skip to use default settings):")
    uploaded = files.upload()
    for fn, content in uploaded.items():
        if fn.endswith('.json'):
            try:
                payload = json.loads(content.decode('utf-8'))
                with open('payload.json', 'w') as f:
                    json.dump(payload, f, indent=2)
                print(f"✅ Successfully loaded and set payload from '{fn}'!")
                break
            except Exception as err:
                print(f"⚠️ Error parsing uploaded JSON: {err}")
except ImportError:
    # Running outside Colab
    if os.path.exists('payload.json'):
        with open('payload.json', 'r') as f:
            payload = json.load(f)
        print("✅ Loaded existing payload.json from workspace.")

# Fallback check for local payload.json if not yet loaded
if not payload and os.path.exists('payload.json'):
    try:
        with open('payload.json', 'r') as f:
            payload = json.load(f)
        print("✅ Loaded payload from local payload.json!")
    except Exception:
        pass

if payload:
    w = payload.get('video', {}).get('width', 1280)
    h = payload.get('video', {}).get('height', 720)
    fps = payload.get('video', {}).get('fps', 30)
    fmt = payload.get('video', {}).get('format', 'mp4')
    style = payload.get('settings', {}).get('style', 'default')
    print(f"🎯 Payload ready: {style} style, {w}×{h} @ {fps}fps ({fmt})")
else:
    print("ℹ️ No JSON uploaded — renderer will use its built-in default settings.")

## 3. Render Video via Headless API
Submits the payload to `/api/render-video`. The server analyzes the audio, renders frames with hardware acceleration, encodes them with FFmpeg, and streams the video back in real-time.

In [ ]:
# Ensure payload is always defined
if 'payload' not in globals() or payload is None:
    if os.path.exists('payload.json'):
        with open('payload.json', 'r') as f:
            payload = json.load(f)
    else:
        payload = {}

video_format = payload.get('video', {}).get('format', 'mp4') if isinstance(payload, dict) else 'mp4'
OUTPUT_FILENAME = f"waveform_render.{video_format}"
render_url = f"{API_URL}/api/render-video"

print(f"🚀 Submitting render request to {render_url}...")
start_time = time.time()

try:
    response = requests.post(render_url, json=payload, stream=True, timeout=600)
    
    if response.status_code == 200:
        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024 * 1024  # 1MB chunks
        
        with open(OUTPUT_FILENAME, 'wb') as f, tqdm(
            desc=OUTPUT_FILENAME,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            for chunk in response.iter_content(chunk_size=block_size):
                if chunk:
                    size = f.write(chunk)
                    bar.update(size)
                    
        elapsed = time.time() - start_time
        file_size_mb = os.path.getsize(OUTPUT_FILENAME) / (1024 * 1024)
        print(f"\n🎉 Render successfully completed in {elapsed:.1f} seconds!")
        print(f"📁 Output saved to: {OUTPUT_FILENAME} ({file_size_mb:.2f} MB)")
    else:
        print(f"❌ Render failed with HTTP status {response.status_code}: {response.text}")
except Exception as e:
    print(f"❌ Request failed: {e}")

## 4. Preview & Playback Video
Preview the generated video directly within Colab.

In [ ]:
if os.path.exists(OUTPUT_FILENAME):
    print("Previewing rendered video:")
    display(Video(OUTPUT_FILENAME, embed=True, width=720))
else:
    print(f"File {OUTPUT_FILENAME} does not exist yet. Run Section 3 first.")

## 5. Download Video to Local Machine
Download the finished video directly to your computer.

In [ ]:
try:
    from google.colab import files
    if os.path.exists(OUTPUT_FILENAME):
        print(f"Downloading {OUTPUT_FILENAME}...")
        files.download(OUTPUT_FILENAME)
    else:
        print(f"File {OUTPUT_FILENAME} not found.")
except ImportError:
    print(f"Not running in Google Colab. File is saved at: {os.path.abspath(OUTPUT_FILENAME)}")